In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development.

In [3]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development.
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)
data.head()


(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


### Let's divide the training and test set into two partitions

In [4]:

from sklearn.model_selection import train_test_split

X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(800,)
(200,)


## Data Preprocessing

In [5]:
import string
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [6]:
import re
# Cleaning function
def clean_html(text):

    # Remove JavaScript
    text = re.sub(r'<script.*?>.*?</script>', '', text, flags=re.DOTALL)

    # Remove CSS
    text = re.sub(r'<style.*?>.*?</style>', '', text, flags=re.DOTALL)

    # Remove comments
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)

    return text

# Apply cleaning
data["clean_text"] = data["text"].apply(clean_html)

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters

- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [7]:
import re

def clean_email(text):
    # Remove inline JavaScript
    text = re.sub(r'<script.*?>.*?</script>', '', text, flags=re.DOTALL)

    # Remove inline CSS
    text = re.sub(r'<style.*?>.*?</style>', '', text, flags=re.DOTALL)

    # Remove HTML comments
    text = re.sub(r'<!--.*?-->', '', text, flags=re.DOTALL)

    # Remove remaining HTML tags
    text = re.sub(r'<[^>]+>', '', text)

    # Remove underscores
    text = re.sub(r'_', ' ', text)

    # Remove non-alphabetic words
    text = re.sub(r'\b[^a-zA-Z]+\b', ' ', text)

    # Remove special characters
    text = re.sub(r'\W', ' ', text)

    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # Remove all single characters
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)

    # Remove single characters from the start
    text = re.sub(r'^[a-zA-Z]\s+', ' ', text)

    # Remove prefixed b
    text = re.sub(r'^b\s+', '', text)

    # Substitute multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)

    # Convert to lowercase
    text = text.lower()

    return text.strip()

In [8]:
#apply the function
data["clean_text"] = data["clean_text"].apply(clean_email)

## Now let's work on removing stopwords
Remove the stopwords.

In [9]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):

    words = text.split()

    filtered_words = []

    for word in words:
        if word not in stop_words:
            filtered_words.append(word)

    return " ".join(filtered_words)

In [10]:
data["clean_text"] = data["clean_text"].apply(remove_stopwords)

In [11]:
data.head()

,text,label,clean_text
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1,dear sir strictly private business proposal mi...
1,Will do.,0,
2,Nora--Cheryl has emailed dozens of memos about...,0,nora cheryl emailed dozens memos haiti weekend...
3,Dear Sir=2FMadam=2C I know that this proposal ...,1,dear sir fmadam know proposal might surprise e...
4,fyi,0,fyi


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [12]:
nltk.download('averaged_perceptron_tagger_eng')
nltk.download("wordnet")
nltk.download("omw-1.4")
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [13]:
def get_wordnet_pos(word):

    # Get NLTK POS tag
    tag = pos_tag([word])[0][1]

    # Convert to WordNet format
    if tag.startswith('J'):
        return wordnet.ADJ

    elif tag.startswith('V'):
        return wordnet.VERB

    elif tag.startswith('N'):
        return wordnet.NOUN

    elif tag.startswith('R'):
        return wordnet.ADV

    else:
        return wordnet.NOUN

In [14]:
def lemmatize_text(text):

    words = text.split()

    lemmatized_words = []

    for word in words:

        pos = get_wordnet_pos(word)

        lemma = lemmatizer.lemmatize(word, pos=pos)

        lemmatized_words.append(lemma)

    return " ".join(lemmatized_words)

In [15]:
data["clean_text"] = data["clean_text"].apply(lemmatize_text)

In [16]:
data["clean_text"].head(10)

,clean_text
0,dear sir strictly private business proposal mi...
1,
2,nora cheryl email dozen memo haiti weekend ple...
3,dear sir fmadam know proposal might surprise e...
4,fyi
5,sure bottom line need special security code ge...
6,dear sir engr ugo nzego engineering store depa...
7,abedin huma saturday november pmhfw quint fmsi...
8,oct th george marshall event department tentat...
9,account owner colleague set aside defray incid...


In [17]:
(data["clean_text"] == "").sum()

np.int64(17)

In [18]:
data = data[data["clean_text"].str.strip() != ""]

In [19]:
data["clean_text"].head(10)

,clean_text
0,dear sir strictly private business proposal mi...
2,nora cheryl email dozen memo haiti weekend ple...
3,dear sir fmadam know proposal might surprise e...
4,fyi
5,sure bottom line need special security code ge...
6,dear sir engr ugo nzego engineering store depa...
7,abedin huma saturday november pmhfw quint fmsi...
8,oct th george marshall event department tentat...
9,account owner colleague set aside defray incid...
10,strong http www cnn com world africa kenya cra...


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [20]:
data["label"].value_counts()

,count
label,
0,544
1,439


In [21]:
ham = data[data["label"] == 0]
spam = data[data["label"] == 1]

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

ham_bow = vectorizer.fit_transform(ham["clean_text"])

ham_word_counts = ham_bow.sum(axis=0)

In [22]:
words = vectorizer.get_feature_names_out()


In [23]:
word_counts = []

for word, count in zip(words, ham_word_counts.tolist()[0]):
    word_counts.append((word, count))

In [24]:
top_ham_words = sorted(word_counts, key=lambda x: x[1], reverse=True)

In [25]:
print(top_ham_words[:10])

[('state', 136), ('pm', 127), ('call', 115), ('say', 111), ('work', 107), ('would', 107), ('president', 99), ('time', 95), ('mr', 91), ('obama', 84)]


In [26]:
# ---------- SPAM ----------

vectorizer = CountVectorizer()

spam_bow = vectorizer.fit_transform(spam["clean_text"])

spam_word_counts = spam_bow.sum(axis=0)

spam_words = vectorizer.get_feature_names_out()

spam_counts = []

for word, count in zip(spam_words, spam_word_counts.tolist()[0]):
    spam_counts.append((word, count))

top_spam = sorted(spam_counts, key=lambda x: x[1], reverse=True)

print("\nTop 10 Spam Words:")
print(top_spam[:10])


Top 10 Spam Words:
[('money', 981), ('account', 895), ('bank', 801), ('fund', 785), ('transfer', 572), ('transaction', 549), ('business', 514), ('country', 508), ('mr', 489), ('nbsp', 475)]


## Extra features

In [27]:
# Split dataframe
data_train, data_val = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

# Indicators
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])

suspicious_words = "|".join([
    "free",
    "cheap",
    "sex",
    "money",
    "account",
    "bank",
    "fund",
    "transfer",
    "transaction",
    "win",
    "deposit",
    "password"
])

# Create features
data_train['money_mark'] = data_train['clean_text'].str.contains(money_simbol_list)*1

data_train['suspicious_words'] = data_train['clean_text'].str.contains(suspicious_words)*1

data_train['text_len'] = data_train['clean_text'].apply(lambda x: len(x))

# Validation set
data_val['money_mark'] = data_val['clean_text'].str.contains(money_simbol_list)*1

data_val['suspicious_words'] = data_val['clean_text'].str.contains(suspicious_words)*1

data_val['text_len'] = data_val['clean_text'].apply(lambda x: len(x))

data_train.head()

,text,label,clean_text,money_mark,suspicious_words,text_len
547,Importance: HighFYI,0,importance highfyi,0,0,18
343,I forgot to tell you- I fixed in your blackber...,0,forgot tell fix blackberry today outdated address,0,0,49
902,processFYI Ã¢ÂÂ we're getting a copy to your...,0,processfyi get copy house well,0,0,30
539,Thank you for all you did on behalf of the Afg...,0,thank behalf afghan woman whirlwind visit emai...,0,1,2362
338,Do you know her?,0,know,0,0,4


## How would you create a Bag of Words with the CountVectorizer method?

In [28]:
def remove_weird_chars(text):
    text = text.encode("ascii", "ignore").decode()
    return text

In [29]:
data_train["clean_text"] = data_train["clean_text"].apply(remove_weird_chars)
data_val["clean_text"] = data_val["clean_text"].apply(remove_weird_chars)

In [34]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

vectorizer = CountVectorizer(
    min_df=3,
    max_df=0.95,
    stop_words="english",
    token_pattern=r"\b[a-zA-Z]{3,}\b"
)

X_train_bow = vectorizer.fit_transform(data_train["clean_text"])

bow_df = pd.DataFrame(
    X_train_bow.toarray(),
    columns=vectorizer.get_feature_names_out()
)

bow_df.head()

,abacha,abandon,abbas,abedin,abidjan,ability,able,abroad,abroadwhere,absence,...,yourpositive,youto,zimbabwe,zimbabwean,zone,zsb,zuma,zwx,zxj,zywn
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [36]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95,
    stop_words="english",
    token_pattern=r"\b[a-zA-Z]{3,}\b"
)

# Vectorize dataset
X_tfidf = tfidf.fit_transform(data["clean_text"])

# Print shape
print(X_tfidf.shape)

(983, 5349)


In [37]:
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf.get_feature_names_out()
)

tfidf_df.head()

,aaa,aacute,aba,abacha,abad,abandon,abank,abbas,abbass,abdul,...,zwqgdg,zwqgew,zwqgu,zwx,zxigcmvhbgx,zxj,zxjlzcb,zxr,zxt,zywn
0,0.0,0.0,0.0,0.0,0.0,0.073904,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Extra Task (optional) - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

Use a MultinimialNB with default parameters.

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [32]:
# Your code